# 固有表現を Wikidata のエンティティに結び付ける (エンティティリンキング)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/16_semantics/entity_linking.ipynb)

NLP クックブック 16-08 ／ 意味解析

文中の「アップル」が企業なのか果物なのか、どのエンティティなのかを特定して、知識ベースの ID に結び付けたい。

In [ ]:
import requests
from sentence_transformers import SentenceTransformer

API = "https://www.wikidata.org/w/api.php"
HEADERS = {"User-Agent": "nlp-cookbook-example/0.1 (educational notebook)"}
model = SentenceTransformer("intfloat/multilingual-e5-small")

def candidates(mention, lang="ja", limit=8):
    """表層形から候補のエンティティを検索する (候補生成)"""
    params = {"action": "wbsearchentities", "search": mention, "language": lang, "uselang": lang, "format": "json", "limit": limit}
    hits = requests.get(API, params=params, headers=HEADERS, timeout=30).json()["search"]
    return [{"id": h["id"], "label": h.get("label", ""), "description": h.get("description", "")} for h in hits]

def link(mention, context, lang="ja"):
    """候補の説明文と文脈の類似度で、最もふさわしいエンティティを選ぶ (曖昧性解消)"""
    cands = [c for c in candidates(mention, lang) if c["description"]]
    if not cands:
        return None
    emb = model.encode([f"passage: {c['label']}: {c['description']}" for c in cands], normalize_embeddings=True)
    sims = emb @ model.encode("query: " + context, normalize_embeddings=True)
    best = int(sims.argmax())
    return {**cands[best], "score": round(float(sims[best]), 3), "url": f"https://www.wikidata.org/wiki/{cands[best]['id']}"}

for mention, context in [("アップル", "アップルは新しい iPhone を発表した。"), ("アップル", "アップルパイを焼くために、アップルを 3 個買った。"),
                         ("ジャガー", "ジャガーは南米の密林に生息するネコ科の動物だ。"), ("ジャガー", "彼はジャガーの新型セダンを購入した。")]:
    print(mention, "|", context, "\n   →", link(mention, context))

In [ ]:
# 結び付けたエンティティの情報を取得する (別名、分類、他言語のラベル)
def entity(qid, lang="ja"):
    params = {"action": "wbgetentities", "ids": qid, "languages": f"{lang}|en", "props": "labels|aliases|claims", "format": "json"}
    e = requests.get(API, params=params, headers=HEADERS, timeout=30).json()["entities"][qid]
    instance_of = [c["mainsnak"]["datavalue"]["value"]["id"] for c in e["claims"].get("P31", [])]
    return {"ja": e["labels"].get("ja", {}).get("value"), "en": e["labels"].get("en", {}).get("value"),
            "aliases": [a["value"] for a in e["aliases"].get(lang, [])][:5], "instance_of": instance_of}

print(entity("Q312"))

- エンティティリンキングは「候補生成」と「曖昧性解消」の 2 段階で行うのが基本。候補生成で正解が漏れると、後段では取り返せない。
- 候補の人気度 (その表層形がどのエンティティを指すことが多いか) は非常に強い手がかり。検索 API の返す順位がほぼそれに当たるので、類似度と組み合わせるとよい。
- 知識ベースにないエンティティ (NIL) の扱いも重要。類似度が低い場合は「該当なし」とする。
- API を連続して呼ぶときは間隔を空ける。大量に処理するなら、Wikidata のダンプや、学習済みのエンティティリンキングモデル (mGENRE、BLINK、ReFinED など) を使う。